In [1]:
import cogsworth
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import pandas as pd

import sys
from importlib import reload

%config InlineBackend.figure_format = 'retina'


pd.options.display.max_columns = 999

plt.rc('font', family='serif')
plt.rcParams['text.usetex'] = False
fs = 24

# update various fontsizes to match
params = {'figure.figsize': (12, 8),
          'legend.fontsize': 0.7*fs,
          'legend.title_fontsize': 0.8*fs,
          'axes.labelsize': fs,
          'xtick.labelsize': 0.9 * fs,
          'ytick.labelsize': 0.9 * fs,
          'axes.linewidth': 1.1,
          'xtick.major.size': 7,
          'xtick.minor.size': 4,
          'ytick.major.size': 7,
          'ytick.minor.size': 4}
plt.rcParams.update(params)

In [2]:
p = cogsworth.pop.load("/mnt/ceph/users/twagg/underworld/sims/template/template_part0.h5")

In [3]:
def get_accreted_mass(group_df, col):
    mass_diff = np.diff(group_df[col].values)
    mass_diff = mass_diff[mass_diff > 0]

    return np.sum(mass_diff) / group_df[col].values[0]

In [4]:
# for each bin_num, find the total difference between each row of mass_1, discarding cases where mass_1 decreases from the previous row
mass_1_accreted = p.bpp.groupby("bin_num").apply(get_accreted_mass, col="mass_1", include_groups=False)
mass_2_accreted = p.bpp.groupby("bin_num").apply(get_accreted_mass, col="mass_2", include_groups=False)
had_rlof = np.isin(p.bin_nums, p.bpp[p.bpp["evol_type"] == 3]["bin_num"].unique())

In [5]:
((~had_rlof) & (mass_1_accreted < 0.05) & (mass_2_accreted < 0.05)).sum() / len(p)

np.float64(0.15492788992098475)

In [6]:
(had_rlof | (mass_1_accreted > 0.05) | (mass_2_accreted > 0.05)).sum() / len(p)

np.float64(0.8450721100790153)